# 08 Audit of saved results

Checks the saved results of notebooks 05 to 07 without retraining:
1. Every saved window is exactly 15 minutes ahead, and rebuilding the windows with an explicit endpoint check gives the same windows.
2. The broad test set restores only readings whose sole problem is the spike rule. If any windows change, the `test_broad` predictions are regenerated from the saved checkpoints.
3. `review/experiment.json` is written and every checkpoint and prediction file is stamped with the experiment ID.

Notebook 09 has to be re-run afterwards.

In [ ]:
!pip install -q datasets pvlib
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/UROP'
LOCAL_CACHE = '/content/skippd_cache'

In [ ]:
import os, shutil, time, math, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
P_DIR = f'{OUT}/review'
for d in [P_DIR, f'{P_DIR}/ckpt', f'{P_DIR}/preds', f'{P_DIR}/figs', f'{OUT}/cache', LOCAL_CACHE]:
    os.makedirs(d, exist_ok=True)
SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
INPUT_LEN, HORIZON = 15, 15
LAT, LON, ALT = 37.4275, -122.1697, 23
PANEL_TILT, PANEL_AZIMUTH, PANEL_KW_DC = 22.5, 195, 30.1
PV_CAP_PCT = 99.95
SEL_C = dict(geo=True, cs=True, kout=True, var=True, resid=True)
TRAIN = dict(batch=256, lr=1e-3, max_epochs=25, patience=4)
DIFF = dict(T=100, max_epochs=60, patience=6, lr=2e-4, batch=512, n_samples=50)
SPLIT_FRAC = dict(train=0.70, select=0.15, calib=0.15)
TIME_TOL_S = 1.0
SPIKE_KW = 8.0
QUANTILES = [0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975]

In [ ]:
def load_split(split):
    names = ['img', 'pv', 'ts']
    local = {k: f'{LOCAL_CACHE}/{split}_{k}.npy' for k in names}
    drive_ = {k: f'{OUT}/cache/{split}_{k}.npy' for k in names}
    if not all(os.path.exists(p) for p in local.values()) and all(os.path.exists(p) for p in drive_.values()):
        print(f'[{split}] copying cache from Drive to local disk...')
        for k in names: shutil.copy(drive_[k], local[k])
    if all(os.path.exists(p) for p in local.values()):
        print(f'[{split}] loading from cache')
        img = np.load(local['img']); pv = np.load(local['pv']); ts_raw = np.load(local['ts'])
    else:
        from datasets import load_dataset
        print(f'[{split}] downloading from Hugging Face (first run only)...')
        ds = load_dataset('solarbench/SKIPPD', split=split)
        n = len(ds); img = np.empty((n, 64, 64, 3), dtype=np.uint8); pos = 0
        for b in ds.iter(batch_size=2000):
            arr = np.stack([np.asarray(im.convert('RGB')) for im in b['image']]); img[pos:pos + len(arr)] = arr; pos += len(arr)
        pv = np.asarray(ds['pv'], dtype=np.float32)
        t = pd.to_datetime(ds['time'])
        if t.tz is not None: t = t.tz_convert('UTC').tz_localize(None)
        ts_raw = t.values.astype('datetime64[ns]')
        for k, v in [('img', img), ('pv', pv), ('ts', ts_raw)]:
            np.save(local[k], v); shutil.copy(local[k], drive_[k])
    order = np.argsort(ts_raw, kind='stable')
    return dict(name=split, img=img, img_idx=order, pv=pv[order].astype(np.float32), ts=pd.DatetimeIndex(ts_raw[order]))

In [ ]:
TR = load_split('train'); TE = load_split('test')
P = np.load(f'{OUT}/day1b_prep.npz', allow_pickle=True)
TR['good'], TE['good'] = P['tr_good'].astype(bool), P['te_good'].astype(bool)
print('train rows', len(TR['pv']), '| test rows', len(TE['pv']))

In [ ]:
from pvlib.location import Location
from pvlib import irradiance
SITE = Location(LAT, LON, altitude=ALT)
def solar_terms(ts_naive):
    a = pd.DatetimeIndex(ts_naive).tz_localize('UTC')
    cs = SITE.get_clearsky(a, model='ineichen'); sp = SITE.get_solarposition(a)
    poa = irradiance.get_total_irradiance(PANEL_TILT, PANEL_AZIMUTH, dni=cs['dni'], ghi=cs['ghi'], dhi=cs['dhi'],
                                          solar_zenith=sp['apparent_zenith'], solar_azimuth=sp['azimuth'])
    return (np.clip(PANEL_KW_DC * poa['poa_global'].values / 1000.0, 0, PANEL_KW_DC).astype(np.float32),
            sp['apparent_zenith'].values.astype(np.float32), sp['azimuth'].values.astype(np.float32))
def local_date(ts): return np.array((pd.DatetimeIndex(ts) - pd.Timedelta(hours=8)).date)
def is_dst(d): return bool((pd.Timestamp(d) + pd.Timedelta(hours=12)).tz_localize('America/Los_Angeles').dst())
CLOCK_FIX = json.load(open(f'{OUT}/clock_fix.json')); print('clock fix:', CLOCK_FIX)
def solar_time(ts):
    ts = pd.DatetimeIndex(ts); ld = local_date(ts); lut = {d: is_dst(d) for d in set(ld)}
    shift = np.where(np.array([lut[d] for d in ld]), CLOCK_FIX['dst_min'], CLOCK_FIX['std_min'])
    return ts - pd.to_timedelta(shift, unit='min')
for D in (TR, TE):
    cs, zen, azi = solar_terms(solar_time(D['ts']))
    D['cs_raw'] = np.nan_to_num(cs); D['zen'] = np.nan_to_num(zen, nan=180.0); D['azi'] = np.nan_to_num(azi)
    D['ld'] = local_date(D['ts'])
    doy = D['ts'].dayofyear.values
    D['doy_s'] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32); D['doy_c'] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

In [ ]:
def spike_flags(pv):
    d1 = np.r_[0, np.diff(pv)]; d2 = np.r_[np.diff(pv), 0]
    return (np.abs(d1) > SPIKE_KW) & (np.abs(d2) > SPIKE_KW) & (np.sign(d1) != np.sign(d2))

def build_windows(D, good, exact=True):
    """Windows of INPUT_LEN readings plus a target HORIZON readings later.
       exact=True: every step must be 60 s +/- TIME_TOL_S (so the target is exactly 15 min after issue).
       exact=False: the notebook 01 rule (no step > 1 min, no step <= 0)."""
    s = D['ts'].values.astype('datetime64[ns]').astype(np.int64) / 1e9
    d = np.diff(s)
    bad = (np.abs(d - 60.0) > TIME_TOL_S) if exact else ((d > 60.0 + 1e-6) | (d <= 0))
    cb = np.r_[0, np.cumsum(bad)]
    t = np.arange(INPUT_LEN - 1, len(s) - HORIZON); st, en = t - (INPUT_LEN - 1), t + HORIZON
    ok = (cb[en] - cb[st]) == 0
    cg = np.r_[0, np.cumsum(~good)]; ok &= (cg[en + 1] - cg[st]) == 0
    if exact:
        ok &= np.abs(s[t + HORIZON] - s[t] - 60.0 * HORIZON) <= TIME_TOL_S
    return t[ok]

def other_pv_flags(D, min_elev=5.0, stuck_n=10, cs_ratio=1.6):
    """Quality checks from notebook 01 other than the spike rule: negative, above the rating,
       above cs_ratio x clear-sky, stuck readings in daylight, low sun."""
    pv, cs, zen = D['pv'], D['cs_raw'], D['zen']
    day = zen < 90 - min_elev
    run_id = np.r_[0, np.cumsum(np.diff(pv) != 0)]
    run_len = np.bincount(run_id)[run_id]
    stuck = (run_len >= stuck_n) & day & (pv > 0)
    return (pv < 0) | (pv > PANEL_KW_DC) | ((pv > cs_ratio * cs) & day) | stuck | ~day

def broad_mask(D):
    """Broad set: restore ONLY readings whose sole problem is the spike rule."""
    return D['good'] | (D['spike'] & ~other_pv_flags(D))

def horizon_seconds(D, w):
    s = D['ts'].values.astype('datetime64[ns]').astype(np.int64) / 1e9
    return s[w + HORIZON] - s[w]

In [ ]:
Z = np.load(f'{P_DIR}/windows_and_splits.npz', allow_pickle=True)
W = {k[2:]: Z[k] for k in Z.files if k.startswith('w_')}
DAY_SPLIT = dict(zip(pd.to_datetime(Z['tr_days']).date, Z['tr_day_split']))
TR['split'] = np.array([DAY_SPLIT[d] for d in TR['ld']])
PV_CAP = float(Z['pv_cap'])
for D in (TR, TE):
    D['cs'] = np.minimum(D['cs_raw'], PV_CAP).astype(np.float32)
    D['k'] = np.clip(D['pv'] / np.maximum(D['cs'], 1.0), 0, 1.5).astype(np.float32)
    D['spike'] = spike_flags(D['pv'])
for k, v in W.items(): print(f'{k:>10}: {len(v)} windows')
print(f'PV cap (training days): {PV_CAP:.2f} kW')

## 1. Horizon check

In [ ]:
import glob
rows = []
for k, w in W.items():
    h = horizon_seconds(TE if k.startswith('test') else TR, w)
    rows.append(dict(set=k, windows=len(w), horizon_min_s=h.min(), horizon_max_s=h.max(), not_900s=int((np.abs(h - 900) > TIME_TOL_S).sum())))
HCHK = pd.DataFrame(rows); print(HCHK.to_string(index=False)); HCHK.to_csv(f'{P_DIR}/p5_horizon_check.csv', index=False)
if HCHK.not_900s.sum(): raise SystemExit('some saved windows are not 15 min ahead (see p5_horizon_check.csv).')

WT_new = build_windows(TR, TR['good']); WE_new = build_windows(TE, TE['good'])
old_tr = np.sort(np.concatenate([W['train'], W['select'], W['calib']]))
print('train-split windows identical with the endpoint check:', np.array_equal(np.sort(WT_new), old_tr))
print('test windows identical with the endpoint check:       ', np.array_equal(WE_new, W['test']))
if not (np.array_equal(np.sort(WT_new), old_tr) and np.array_equal(WE_new, W['test'])):
    raise SystemExit('the endpoint check changes the main windows; the saved results cannot be reused.')

## 2. Broad test set

In [ ]:
other = other_pv_flags(TE); sp_bad = TE['spike'] & ~TE['good']
WEB_new = build_windows(TE, broad_mask(TE)); WEB_old = W['test_broad']
BCHK = dict(spike_flagged_bad_readings=int(sp_bad.sum()), of_which_fail_another_check=int((sp_bad & other).sum()),
            broad_windows_old=len(WEB_old), broad_windows_new=len(WEB_new),
            dropped=len(np.setdiff1d(WEB_old, WEB_new)), added=len(np.setdiff1d(WEB_new, WEB_old)))
print(BCHK); pd.DataFrame([BCHK]).to_csv(f'{P_DIR}/p5_broad_check.csv', index=False)
BROAD_CHANGED = not np.array_equal(WEB_new, WEB_old)
if BROAD_CHANGED:
    os.makedirs(f'{P_DIR}/broad_v1', exist_ok=True)
    wpath = f'{P_DIR}/windows_and_splits.npz'
    if not os.path.exists(f'{P_DIR}/broad_v1/windows_and_splits.npz'): shutil.copy(wpath, f'{P_DIR}/broad_v1/windows_and_splits.npz')
    Zd = dict(np.load(wpath, allow_pickle=True)); W['test_broad'] = WEB_new
    Zd['w_test_broad'] = WEB_new
    Zd['issue_test_broad'] = TE['ts'][WEB_new].values.astype('datetime64[s]').astype(str)
    Zd['target_test_broad'] = TE['ts'][WEB_new + HORIZON].values.astype('datetime64[s]').astype(str)
    Zd['y_test_broad'] = TE['pv'][WEB_new + HORIZON]
    np.savez_compressed(wpath, **Zd); print('broad windows updated; old file kept in review/broad_v1/')
else:
    print('broad windows unchanged: nothing to regenerate')

## 3. Experiment ID

In [ ]:
MIGRATE = True

In [ ]:
MODEL_VERSIONS = dict(cnn_mse='v1', cnn_gauss='v1', cnn_quant='v2-sorted', diffusion='v1')
def experiment_id():
    h = hashlib.sha256()
    for k in sorted(W): h.update(k.encode()); h.update(np.ascontiguousarray(W[k]).astype(np.int64).tobytes())
    cfg = dict(TRAIN=TRAIN, DIFF=DIFF, SPLIT_FRAC=SPLIT_FRAC, TIME_TOL_S=TIME_TOL_S, SPIKE_KW=SPIKE_KW, QUANTILES=QUANTILES,
               INPUT_LEN=INPUT_LEN, HORIZON=HORIZON, PV_CAP=round(float(PV_CAP), 4), MODEL_VERSIONS=MODEL_VERSIONS,
               DAY_SPLIT={str(d): str(v) for d, v in sorted(DAY_SPLIT.items())})
    h.update(json.dumps(cfg, sort_keys=True).encode()); return h.hexdigest()[:16]
EXP_ID = experiment_id(); _ep = f'{P_DIR}/experiment.json'
def _write_exp(): json.dump(dict(exp_id=EXP_ID, written=pd.Timestamp.now().isoformat(timespec='seconds'), model_versions=MODEL_VERSIONS), open(_ep, 'w'), indent=2)
if globals().get('MIGRATE', False):
    _write_exp(); print('experiment.json written by migration (notebook 08)')
elif os.path.exists(_ep):
    _old = json.load(open(_ep))['exp_id']
    if _old != EXP_ID:
        raise SystemExit(f'this notebook builds experiment {EXP_ID}, but review/experiment.json is {_old}. '
                         'Windows, splits, settings or model versions differ from the saved run.')
elif globals().get('CREATE_EXPERIMENT', False):
    _write_exp()
else:
    raise SystemExit('review/experiment.json not found. Run notebook 05 (new experiment) or notebook 08 (existing results) first.')
print('experiment id:', EXP_ID)

In [ ]:
OFF = np.arange(-(INPUT_LEN - 1), 1)
def smart_base(D, t):
    return np.clip(D['pv'][t] / np.maximum(D['cs_raw'][t], 1e-3) * D['cs_raw'][t + HORIZON], 0, PV_CAP).astype(np.float32)

def vec_feats(D, t, c):
    H = t[:, None] + OFF; tg = t + HORIZON
    parts = [D['pv'][H] / PV_CAP]
    if c.get('geo'):
        parts.append(np.stack([D['doy_s'][t], D['doy_c'][t], D['zen'][t] / 90, D['zen'][tg] / 90,
                               np.cos(np.radians(D['zen'][t])), np.sin(np.radians(D['azi'][t])), np.cos(np.radians(D['azi'][t]))], 1))
    if c.get('cs'): parts += [D['cs'][H] / PV_CAP, (D['cs'][tg] / PV_CAP)[:, None]]
    if c.get('kout'): parts.append(D['k'][H])
    if c.get('var'):
        kh, ph = D['k'][H], D['pv'][H]
        parts.append(np.stack([kh.std(1), np.abs(np.diff(ph, axis=1)).mean(1) / PV_CAP,
                               (ph[:, -1] - ph[:, -6]) / PV_CAP, (ph.max(1) - ph.min(1)) / PV_CAP], 1))
    if c.get('resid'): parts.append((smart_base(D, t) / PV_CAP)[:, None])
    return np.concatenate(parts, 1).astype(np.float32)

def to_img(D, t):
    return torch.from_numpy(D['img'][D['img_idx'][t]]).to(device).permute(0, 3, 1, 2).float().div_(255.0)

def batches(D, w, c, bs, shuffle, rng=None):
    idx = rng.permutation(len(w)) if shuffle else np.arange(len(w))
    for i in range(0, len(w), bs):
        t = w[idx[i:i + bs]]
        xv = torch.from_numpy(vec_feats(D, t, c)).to(device)
        img = to_img(D, t)
        y = torch.from_numpy(D['pv'][t + HORIZON] / PV_CAP).float().to(device)
        yield xv, img, y

N_VEC = vec_feats(TR, W['train'][:2], SEL_C).shape[1]

def conv_branch(img_ch=3):
    return nn.Sequential(nn.Conv2d(img_ch, 16, 5, stride=2, padding=2), nn.ReLU(),
                         nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                         nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4))

In [ ]:
def _pred_path(name, seed): return f'{P_DIR}/preds/{name}_seed{seed}.npz'
def pred_exp_id(name, seed):
    return json.loads(str(np.load(_pred_path(name, seed), allow_pickle=True)['meta'])).get('exp_id')
def save_preds(name, seed, arrays, meta=None):
    p = _pred_path(name, seed)
    np.savez_compressed(p, **arrays, meta=json.dumps({**(meta or {}), 'exp_id': EXP_ID}))
    return p
def have_preds(name, seed):
    if not os.path.exists(_pred_path(name, seed)): return False
    e = pred_exp_id(name, seed)
    if e != EXP_ID: raise SystemExit(f'{name} seed {seed}: saved for experiment {e}, current is {EXP_ID}. Move or delete it, or run notebook 08.')
    return True
def load_preds(name, seed):
    assert have_preds(name, seed), f'missing predictions: {name} seed {seed}'
    return dict(np.load(_pred_path(name, seed), allow_pickle=True))
EVAL_SETS = ['select', 'calib', 'test', 'test_broad']
def data_of(k): return TE if k.startswith('test') else TR

In [ ]:
class CorrNet(nn.Module):
    """Correction CNN with a configurable head. Output is in units of PV/PV_CAP.
       kind='mse'  : 1 output, forecast = smart persistence + correction
       kind='gauss': 2 outputs, mean = smart persistence + correction, variance = softplus(raw) + floor
       kind='quant': len(QUANTILES) outputs = smart persistence + sorted corrections (non-crossing)"""
    def __init__(self, kind, use_image=True, p_drop=0.2):
        super().__init__()
        self.kind, self.use_image = kind, use_image
        if use_image: self.conv = conv_branch()
        self.vec = nn.Sequential(nn.Linear(N_VEC, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        n_out = {'mse': 1, 'gauss': 2, 'quant': len(QUANTILES)}[kind]
        self.head = nn.Sequential(nn.Linear((32 * 16 if use_image else 0) + 32, 64), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(64, n_out))
    def forward(self, xv, img):
        f = self.vec(xv)
        if self.use_image: f = torch.cat([self.conv(img).flatten(1), f], 1)
        o = self.head(f); sp = xv[:, -1]
        if self.kind == 'mse': return sp + o[:, 0]
        if self.kind == 'gauss': return sp + o[:, 0], F.softplus(o[:, 1]) + 1e-6
        return sp[:, None] + torch.sort(o, 1).values

QT = torch.tensor(QUANTILES, device=device)
def loss_of(kind, out, y):
    if kind == 'mse': return F.mse_loss(out, y)
    if kind == 'gauss': mu, var = out; return F.gaussian_nll_loss(mu, y, var, full=False, eps=1e-6)
    e = y[:, None] - out; return torch.maximum(QT * e, (QT - 1) * e).mean()

def train_cnn(kind, seed, use_image=True):
    tag = f'{kind}{"" if use_image else "_noimg"}_seed{seed}'; ck = f'{P_DIR}/ckpt/p1_{tag}.pt'
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = CorrNet(kind, use_image).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=TRAIN['lr'], weight_decay=1e-4)
    st = dict(epoch=0, best=float('inf'), bad=0, hist=[], best_state=None, done=False, train_sec=0.0)
    if os.path.exists(ck):
        st = torch.load(ck, map_location=device, weights_only=False)
        if st.get('exp_id') != EXP_ID: raise SystemExit(f'checkpoint {ck} belongs to experiment {st.get("exp_id")}, current is {EXP_ID}.')
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
        if not st['done']: print(f'[{tag}] resuming from epoch {st["epoch"]}')
    st['exp_id'] = EXP_ID
    while not st['done']:
        t0 = time.time(); model.train(); tl = []
        for xv, img, y in batches(TR, W['train'], SEL_C, TRAIN['batch'], True, rng):
            loss = loss_of(kind, model(xv, img), y); opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval(); vs, vn = 0.0, 0
        with torch.no_grad():
            for xv, img, y in batches(TR, W['select'], SEL_C, 2048, False):
                vs += loss_of(kind, model(xv, img), y).item() * len(y); vn += len(y)
        v = vs / vn; st['hist'].append((float(np.mean(tl)), v)); st['epoch'] += 1; st['train_sec'] += time.time() - t0
        if v < st['best']: st['best'], st['bad'] = v, 0; st['best_state'] = {k: x.detach().clone() for k, x in model.state_dict().items()}
        else: st['bad'] += 1
        st['done'] = st['epoch'] >= TRAIN['max_epochs'] or st['bad'] >= TRAIN['patience']
        st['model'], st['opt'] = model.state_dict(), opt.state_dict(); torch.save(st, ck)
        print(f'[{tag}] epoch {st["epoch"]:2d}  train {np.mean(tl):.5f}  select {v:.5f}  ({time.time()-t0:.0f}s)')
    model.load_state_dict(st['best_state']); model.eval()
    return model, st

@torch.no_grad()
def predict_cnn(model, D, w):
    outs = []
    for xv, img, _ in batches(D, w, SEL_C, 2048, False):
        o = model(xv, img)
        if model.kind == 'gauss': o = torch.stack([o[0], o[1].sqrt()], 1)
        elif model.kind == 'mse': o = o[:, None]
        outs.append(o.cpu().numpy())
    return np.concatenate(outs) * PV_CAP

In [ ]:
def cosine_betas(T, s=0.008):
    x = torch.linspace(0, T, T + 1)
    ac = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2; ac = ac / ac[0]
    return torch.clip(1 - ac[1:] / ac[:-1], 1e-4, 0.999)
BETAS = cosine_betas(DIFF['T']).to(device); ALPHAS = 1 - BETAS; ACP = torch.cumprod(ALPHAS, 0)

def t_embed(t, dim=16):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float()[:, None] * f[None]
    return torch.cat([torch.sin(a), torch.cos(a)], -1)

class DiffNet(nn.Module):
    """notebook 04 design. use_image=False drops the image branch (retrained, not blanked)."""
    def __init__(self, use_image=True):
        super().__init__()
        self.use_image = use_image
        if use_image: self.conv = conv_branch()
        self.vec = nn.Sequential(nn.Linear(N_VEC, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.den = nn.Sequential(nn.Linear(1 + 16 + (32 * 16 if use_image else 0) + 32, 256), nn.SiLU(),
                                 nn.Linear(256, 128), nn.SiLU(), nn.Linear(128, 1))
    def context(self, xv, img):
        f = self.vec(xv)
        return torch.cat([self.conv(img).flatten(1), f], 1) if self.use_image else f
    def denoise(self, z, t, ctx):
        return self.den(torch.cat([z[:, None], t_embed(t), ctx], 1)).squeeze(-1)

RESID_SD = float((TR['pv'][W['train'] + HORIZON] / PV_CAP - smart_base(TR, W['train']) / PV_CAP).std())
def to_z(y, xv): return (y - xv[:, -1]) / RESID_SD
def from_z(z, xv): return xv[:, -1:] + z * RESID_SD
print(f'correction scale (training days): {RESID_SD:.4f} of the cap = {RESID_SD * PV_CAP:.2f} kW')

def diff_loss(model, xv, img, y, gen=None):
    z0 = to_z(y, xv); B = len(z0)
    t = torch.randint(0, DIFF['T'], (B,), device=device, generator=gen)
    eps = torch.randn(B, device=device, generator=gen)
    zt = ACP[t].sqrt() * z0 + (1 - ACP[t]).sqrt() * eps
    return ((model.denoise(zt, t, model.context(xv, img)) - eps) ** 2).mean()

def diff_tag(seed, use_image): return f'diff{"" if use_image else "_noimg"}_seed{seed}'

def train_diffusion(seed, use_image=True):
    tag = diff_tag(seed, use_image); ck = f'{P_DIR}/ckpt/p2_{tag}.pt'
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = DiffNet(use_image).to(device); opt = torch.optim.AdamW(model.parameters(), lr=DIFF['lr'], weight_decay=1e-4)
    st = dict(epoch=0, best=float('inf'), bad=0, hist=[], best_state=None, done=False, train_sec=0.0)
    if os.path.exists(ck):
        st = torch.load(ck, map_location=device, weights_only=False)
        if st.get('exp_id') != EXP_ID: raise SystemExit(f'checkpoint {ck} belongs to experiment {st.get("exp_id")}, current is {EXP_ID}.')
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
        if not st['done']: print(f'[{tag}] resuming from epoch {st["epoch"]}')
    st['exp_id'] = EXP_ID
    while not st['done']:
        t0 = time.time(); model.train(); tl = []
        for xv, img, y in batches(TR, W['train'], SEL_C, DIFF['batch'], True, rng):
            loss = diff_loss(model, xv, img, y); opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval(); gen = torch.Generator(device=device).manual_seed(123); vs, vn = 0.0, 0
        with torch.no_grad():
            for xv, img, y in batches(TR, W['select'], SEL_C, 4096, False):
                vs += diff_loss(model, xv, img, y, gen).item() * len(y); vn += len(y)
        v = vs / vn; st['hist'].append((float(np.mean(tl)), v)); st['epoch'] += 1; st['train_sec'] += time.time() - t0
        if v < st['best']: st['best'], st['bad'] = v, 0; st['best_state'] = {k: x.detach().clone() for k, x in model.state_dict().items()}
        else: st['bad'] += 1
        st['done'] = st['epoch'] >= DIFF['max_epochs'] or st['bad'] >= DIFF['patience']
        st['model'], st['opt'] = model.state_dict(), opt.state_dict(); torch.save(st, ck)
        print(f'[{tag}] epoch {st["epoch"]:2d}  train {np.mean(tl):.4f}  select {v:.4f}  ({time.time()-t0:.0f}s)')
    model.load_state_dict(st['best_state']); model.eval()
    return model, st

def load_diffusion(seed, use_image=True):
    st = torch.load(f'{P_DIR}/ckpt/p2_{diff_tag(seed, use_image)}.pt', map_location=device, weights_only=False)
    if st.get('exp_id') != EXP_ID: raise SystemExit(f'diffusion checkpoint seed {seed} belongs to experiment {st.get("exp_id")}, current is {EXP_ID}.')
    m = DiffNet(use_image).to(device); m.load_state_dict(st['best_state']); m.eval(); return m, st

@torch.no_grad()
def diff_sample(model, D, w, clip='step', cap=None, n=DIFF['n_samples'], bs=512, seed=0, out_range=None):
    """clip: 'step' = clip the clean-sample estimate to [0, cap] at every reverse step (notebooks 03–04 method)
             'final' = no clipping during sampling, clip only the final PV value
             'none'  = no clipping at all
       The same seed gives the same initial noise and the same random draws for every variant.
       out_range: optional dict to collect how many final (unclipped) samples fell outside [0, cap] and any NaNs."""
    cap = PV_CAP if cap is None else cap
    g = torch.Generator(device=device).manual_seed(seed); out = []
    for xv, img, y in batches(D, w, SEL_C, bs, False):
        ctx = model.context(xv, img); B = len(ctx)
        ctx_r = ctx.repeat_interleave(n, 0); xr = xv.repeat_interleave(n, 0)
        z = torch.randn(B * n, device=device, generator=g)
        lo_z = to_z(torch.zeros(B * n, device=device), xr); hi_z = to_z(torch.full((B * n,), cap / PV_CAP, device=device), xr)
        for k in reversed(range(DIFF['T'])):
            tt = torch.full((B * n,), k, device=device, dtype=torch.long)
            eps = model.denoise(z, tt, ctx_r)
            ac, ac_prev = ACP[k], (ACP[k - 1] if k > 0 else torch.tensor(1.0, device=device))
            x0 = (z - (1 - ac).sqrt() * eps) / ac.sqrt()
            if clip == 'step': x0 = torch.maximum(torch.minimum(x0, hi_z), lo_z)
            mean = (ac_prev.sqrt() * BETAS[k] / (1 - ac)) * x0 + (ALPHAS[k].sqrt() * (1 - ac_prev) / (1 - ac)) * z
            noise = torch.randn(B * n, device=device, generator=g)
            z = mean + (BETAS[k] * (1 - ac_prev) / (1 - ac)).sqrt() * noise if k > 0 else x0
        yhat = from_z(z.view(B, n), xv) * PV_CAP
        if out_range is not None:
            out_range['nan'] = out_range.get('nan', 0) + int(torch.isnan(yhat).sum())
            out_range['outside'] = out_range.get('outside', 0) + int(((yhat < -1e-3) | (yhat > cap + 1e-3)).sum())
            out_range['total'] = out_range.get('total', 0) + yhat.numel()
        if clip in ('step', 'final'): yhat = yhat.clamp(0, cap)
        out.append(yhat.cpu().numpy().astype(np.float32))
    return np.concatenate(out)

In [ ]:
for f in sorted(glob.glob(f'{P_DIR}/ckpt/*.pt')):
    st = torch.load(f, map_location='cpu', weights_only=False)
    if st.get('exp_id') != EXP_ID: st['exp_id'] = EXP_ID; torch.save(st, f)
print('checkpoints stamped:', len(glob.glob(f'{P_DIR}/ckpt/*.pt')))

## 4. Prediction files

In [ ]:
SEEDS = [0, 1, 2]
def ckpt_cnn(kind, s):
    st = torch.load(f'{P_DIR}/ckpt/p1_{kind}_seed{s}.pt', map_location=device, weights_only=False)
    m = CorrNet(kind).to(device); m.load_state_dict(st['best_state']); m.eval(); return m
LOG = []
for f in sorted(glob.glob(f'{P_DIR}/preds/*.npz')):
    name, s = os.path.basename(f)[:-4].rsplit('_seed', 1); s = int(s)
    Zp = dict(np.load(f, allow_pickle=True)); meta = json.loads(str(Zp.pop('meta')))
    for k in [k for k in Zp if k in W and k != 'test_broad']:
        if len(Zp[k]) != len(W[k]): raise SystemExit(f'{name} seed {s} has {len(Zp[k])} rows for {k}, windows have {len(W[k])}')
    regen = BROAD_CHANGED and 'test_broad' in Zp
    if regen:
        t0 = time.time()
        if name.startswith('cnn_'):
            Zp['test_broad'] = predict_cnn(ckpt_cnn(name[4:], s), TE, W['test_broad'])
        elif name in ('diff', 'diff_noimg'):
            m, _ = load_diffusion(s, name == 'diff')
            Zp['test_broad'] = diff_sample(m, TE, W['test_broad'], clip='step', seed=1000 + s)
        else:
            raise SystemExit(f'do not know how to regenerate test_broad for {name}')
        meta['test_broad_regenerated'] = pd.Timestamp.now().isoformat(timespec='seconds')
        print(f'{name} seed {s}: test_broad regenerated ({time.time() - t0:.0f}s)')
    elif 'test_broad' in Zp and len(Zp['test_broad']) != len(W['test_broad']):
        raise SystemExit(f'{name} seed {s} test_broad rows do not match the windows')
    meta['exp_id'] = EXP_ID
    np.savez_compressed(f, **Zp, meta=json.dumps(meta)); LOG.append(dict(file=os.path.basename(f), regenerated_broad=regen))
print(pd.DataFrame(LOG).to_string(index=False))
for name in ['cnn_mse', 'cnn_gauss', 'cnn_quant', 'diff', 'diff_noimg']:
    missing = [s for s in SEEDS if not have_preds(name, s)]
    print(f'{name}: seeds present {[s for s in SEEDS if s not in missing]}' + (f'  MISSING {missing}' if missing else ''))

In [ ]:
dev = f'{P_DIR}/deviations.md'
note = ('P5: exact-horizon endpoint check added (saved windows verified identical); broad test-set mask changed to restore only '
        f'readings whose sole problem is the spike rule (broad windows {BCHK["broad_windows_old"]} -> {BCHK["broad_windows_new"]}); '
        f'experiment ID {EXP_ID} written and stamped on all checkpoints and predictions.')
if not (os.path.exists(dev) and 'P5: exact-horizon endpoint check' in open(dev).read()):
    with open(dev, 'a') as fh: fh.write(f"\n- {pd.Timestamp.now().isoformat(timespec='seconds')}: {note}\n")
print(open(dev).read())
print('Done. Next: re-run notebook 09.')